# Complete Car Price Prediction PipelineIs notebook mein **Linear Regression**, **Ridge**, **Random Forest**, aur **Gradient Boosting** ko compare kiya gaya hai. Saath hi **Encoder**, **Scaler**, aur **User Choice Model** export karne ka option hai.

In [1]:
import numpy as np
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor


### Step 1: Data Cleaning & Outlier Filtering (< 1000 km & Price IQR)

In [2]:
df = pd.read_csv('quikr_car.csv')

# Year cleaning
df = df[df['year'].str.isnumeric()].copy()
df['year'] = df['year'].astype(int)

# Price cleaning
df = df[df['Price'] != 'Ask For Price'].copy()
df['Price'] = df['Price'].str.replace(',', '').astype(int)

# Kms driven cleaning (< 1000 km removed)
df['kms_driven'] = df['kms_driven'].str.split(' ').str.get(0).str.replace(',', '')
df = df[df['kms_driven'].str.isnumeric()].copy()
df['kms_driven'] = df['kms_driven'].astype(int)
df = df[df['kms_driven'] >= 1000].copy()

# Fuel type and Name cleaning
df = df[~df['fuel_type'].isna()].copy()
df['name'] = df['name'].str.split(' ').str.slice(0, 3).str.join(' ')

# Price IQR Outlier Filter
Q1 = df['Price'].quantile(0.25)
Q3 = df['Price'].quantile(0.75)
IQR = Q3 - Q1
df_clean = df[df['Price'] <= (Q3 + 1.5 * IQR)].reset_index(drop=True)

print("Cleaned shape:", df_clean.shape)


Cleaned shape: (746, 6)


### Step 2: Separate Preprocessing (Encoder & Scaler Fit)

In [3]:
X = df_clean.drop(columns='Price')
y = df_clean['Price']
y_log = np.log1p(y)

X_train, X_test, y_train_log, y_test_log = train_test_split(X, y_log, test_size=0.2, random_state=42)

cat_cols = ['name', 'company', 'fuel_type']
num_cols = ['year', 'kms_driven']

# Fit Encoder separately
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_cat = encoder.fit_transform(X_train[cat_cols])
X_test_cat = encoder.transform(X_test[cat_cols])

# Fit Scaler separately
scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train[num_cols])
X_test_num = scaler.transform(X_test[num_cols])

# Combine processed features
X_train_processed = np.hstack((X_train_cat, X_train_num))
X_test_processed = np.hstack((X_test_cat, X_test_num))


### Step 3: Train & Compare 4 Models

In [4]:
# 1. Linear Regression
lr = LinearRegression()
lr.fit(X_train_processed, y_train_log)

# 2. Ridge Regression
ridge_grid = GridSearchCV(Ridge(), {'alpha': [0.1, 1.0, 10.0, 100.0]}, cv=3, scoring='neg_mean_absolute_error')
ridge_grid.fit(X_train_processed, y_train_log)
ridge = ridge_grid.best_estimator_

# 3. Random Forest
rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    {'n_estimators': [100, 200], 'max_depth': [10, 15]},
    cv=3, scoring='neg_mean_absolute_error', n_jobs=-1
)
rf_grid.fit(X_train_processed, y_train_log)
rf = rf_grid.best_estimator_

# 4. Gradient Boosting
gb_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 5]},
    cv=3, scoring='neg_mean_absolute_error', n_jobs=-1
)
gb_grid.fit(X_train_processed, y_train_log)
gb = gb_grid.best_estimator_

models = {
    'Linear Regression': lr,
    'Ridge Regression': ridge,
    'Random Forest': rf,
    'Gradient Boosting': gb
}


### Step 4: Model Performance Evaluation

In [5]:
results = []
y_test_orig = np.expm1(y_test_log)

for name, model in models.items():
    preds_log = model.predict(X_test_processed)
    preds = np.expm1(preds_log)
    
    results.append({
        'Model': name,
        'R2 Score': r2_score(y_test_orig, preds),
        'MAE (₹)': mean_absolute_error(y_test_orig, preds),
        'RMSE (₹)': np.sqrt(mean_squared_error(y_test_orig, preds))
    })

results_df = pd.DataFrame(results).sort_values(by='MAE (₹)').reset_index(drop=True)
print("=== Model Comparison Table ===")
print(results_df)


=== Model Comparison Table ===
               Model  R2 Score       MAE (₹)       RMSE (₹)
0  Linear Regression  0.854983  44685.435569   70119.099910
1   Ridge Regression  0.858097  44931.029862   69362.046641
2  Gradient Boosting  0.798760  58048.151722   82600.561924
3      Random Forest  0.690174  71778.996483  102490.843790


### Step 5: Save Preprocessors & Selected Model for Streamlit

In [6]:
# 1. Save Encoder & Scaler
with open('encoder.pkl', 'wb') as f:
  pickle.dump(encoder, f)

with open('scaler.pkl', 'wb') as f:
  pickle.dump(scaler, f)

# 2. Save Dropdown options list for Streamlit UI
dropdown_options = {
    'names': sorted(df_clean['name'].unique().tolist()),
    'companies': sorted(df_clean['company'].unique().tolist()),
    'fuel_types': sorted(df_clean['fuel_type'].unique().tolist()),
}
with open('dropdown_options.pkl', 'wb') as f:
  pickle.dump(dropdown_options, f)

# 3. Save Linear Regression Model Directly
with open('model.pkl', 'wb') as f:
  pickle.dump(models['Linear Regression'], f)

print(
    'Successfully exported:\n- encoder.pkl\n- scaler.pkl\n-'
    ' dropdown_options.pkl\n- model.pkl (Linear Regression Saved)'
)

Successfully exported:
- encoder.pkl
- scaler.pkl
- dropdown_options.pkl
- model.pkl (Linear Regression Saved)
